# Unit distance graphs

Anton Antonov  
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com)  
May 2026

---

## Setup

In [ ]:
use Math::NumberTheory;
use Math::NumberTheory::Utilities;
use Math::SparseMatrix;

use Math::Nearest;
use Math::DistanceFunctions;

use Graph;
use Graph::Classes;

use Data::Reshapers;
use Graphviz::DOT::Chessboard;

### D3.js

In [ ]:
#%javascript
require.config({
     paths: {
     d3: 'https://d3js.org/d3.v7.min'
}});

require(['d3'], function(d3) {
     console.log(d3);
});

In [ ]:
#%js
js-d3-list-line-plot(10.rand xx 30, background => 'none')

In [ ]:
my $title-color = 'Ivory';
my $stroke-color = 'SlateGray';
my $tooltip-color = 'LightBlue';
my $tooltip-background-color = 'none';
my $background = '#1F1F1F';
my $color-scheme = 'schemeTableau10';
my $edge-thickness = 3;
my $vertex-size = 3;
my $mmd-theme = q:to/END/;
%%{
  init: {
    'theme': 'forest',
    'themeVariables': {
      'lineColor': 'Ivory'
    }
  }
}%%
END
sink my %force = charge => {strength => -30, iterations => 4}, collision => {radius => 50, iterations => 4}, link => {distance => 30};

---

## Introduction

In [ ]:
#% html
Graph::Complete.new([4,4]).dot(engine => 'neato'):svg

In [ ]:
#% html
my @edges = cross('01'.comb, 'abc'.comb);
my %vertex-coordinates = '01abc'.comb Z=> [|cross(0, ^2), |cross(1, (^3) <<->> 0.5)];
Graph.new(:@edges, :%vertex-coordinates).dot(engine => 'neato'):svg

In [ ]:
#% html
my @edges = |cross('01'.comb, 'abc'.comb), |cross('abc'.comb, 'ABCDE'.comb), |cross('01'.comb, 'ABCDE'.comb);
my %vertex-coordinates = '01abcABCDE'.comb Z=> [|cross(0, ^2 <<->> ((2 - 1) / 2)), |cross(1, (^3) <<->> ((3 - 1) / 2)), |cross(2, (^5) <<->> ((5 - 1) / 2))];
#my %vertex-coordinates = '01abcABCDE'.comb Z=> [|cross(0, ^2), |cross(1, (^3)), |cross(2, (^5))];
my $g = Graph.new(:@edges, :%vertex-coordinates);

$g.dot(engine => 'neato'):svg

In [ ]:
#% js
js-d3-graph-plot(
    $g.edges, 
    vertex-coordinates => $g.vertex-coordinates,
    vertex-size => 5,
    :$title-color,
    :$background,
    :%force,
    :500width,
    :500height,
)

----

## Leaper graphs

In order to construct square grid graphs with edges that are of length 1, we consider the family of [*leaper graphs*](https://mathworld.wolfram.com/LeaperGraph.html). The Leaper graph  generalizes the [Knight Tour graph](https://mathworld.wolfram.com/KnightGraph.html). Here are the moves of [Camel graph](https://mathworld.wolfram.com/CamelGraph.html), Flamingo graph,  and [Zebra graph](https://mathworld.wolfram.com/ZebraGraph.html), which are leaper graphs parameterized with $(1, 3)$, $(1, 6)$, and $(2, 3)$, respectively:

In [ ]:
#% html
my %opts-brown = black-square-color => 'SandyBrown', white-square-color => 'Moccasin', :65font-size;
my $fenC = '8/2N1N2/8/N5N1/3n3/N5N1/8/2N2N2';
my $c = dot-chessboard($fenC, :7r, :7c, :4size, background=>'none', |%opts-brown, :svg);
$c .= subst(/ '♞' | '♘'/, '🐪', :g);

my %opts-blue = black-square-color => 'DarkSeaGreen', white-square-color => 'Wheat', :65font-size;
my $fenF = '8/N1N5/8/8/8/6N1/1n6/6N1';
my $f = dot-chessboard($fenF, :7r, :7c, :4size, background=>'none', |%opts-blue, :svg);
$f .= subst(/ '♞' | '♘'/, '🦩', :g);

my %opts-green = black-square-color => '#779556ff', white-square-color => '#ebedb7', :65font-size;
my $fenZ = '8/1N3N1/N5N1/8/3n3/8/N5N1/1N3N1';
my $z = dot-chessboard($fenZ, :7r, :7c, :4size, background=>'none', |%opts-green, :svg);
$z .= subst(/ '♞' | '♘'/, '🦓', :g);


$c ~ $f ~ $z

**Remark:** From the code and plots above it can be seen that the package package ["Graphviz::DOT::Chessboard"](https://raku.land/zef:antononcube/Graphviz::DOT::Chessboard), [AAp5], can handle chess boards and FEN notations with non-standard sizes.

In [ ]:
my @graphs = [(1,3), (1,6), (2,3)].map({ Graph::Leaper.new(moves => $_, rows => 10, columns => 10) });
sink my @mats = @graphs.map({ $_.adjacency-matrix });
deduce-type(@mats)

In [ ]:
#% html
my @graphs = [(1,3), (1,6), (2,3)].map({ Graph::Leaper.new(moves => $_, rows => 10, columns => 10) });
@graphs.map({ dot-matrix-plot($_.adjacency-matrix, :3graph-size, :svg) }).join("\n")

In [ ]:
#% js
my @graphs = [(1,3), (1,6), (2,3)].map({ Graph::Leaper.new(moves => $_, rows => 16, columns => 16) });
my @mats = @graphs.map({ Math::SparseMatrix.new(edges => $_.edges(:dataset)) });
@mats.map({ js-d3-matrix-plot($_.tuples.map({ <x y z>.Array Z=> $_.Array })».Hash, :350width, :350height, :!tooltip, :!grid-lines, :!mesh, color-palette => 'Turbo') }).join("\n")

----

## Fairy chess graph (first)

Let us make a graph which by construction has edges that are of length 1. Consider the family of ["leaper graphs"](https://mathworld.wolfram.com/LeaperGraph.html) that generalize the [Knight Tour graph](https://mathworld.wolfram.com/KnightGraph.html).

We can ask ourselves:

1. Can we construct a "single-pattern" leaper graph in which the edges corresponding to all leaps are of length 1?
2. Can we combine a few leaper graphs in order to produce a unit distance graph?

In [ ]:
#% html
my $fen = '8/5N1/N7/8/3n3/8/8/5N1';
my $z = dot-chessboard($fen, :7r, :7c, :4size, background=>'none', :svg);
$z .= subst(/'<!-- ' .*? ' -->' \n/, :g);
$z.subst(/ '♞' | '♘'/, '🦓', :g)

In [ ]:
#% html
my $zebra-graph = q:to/ZEBRA_END/;
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 520 520" width="520" height="520">
  <defs>
    <marker id="arrow" markerWidth="10" markerHeight="10" refX="7" refY="5" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L10,5 L0,10 z" fill="#c7d2fe"/>
    </marker>
    <filter id="glow" x="-30%" y="-30%" width="160%" height="160%">
      <feDropShadow dx="0" dy="0" stdDeviation="4" flood-color="#7c8cff" flood-opacity="0.45"/>
    </filter>
    <style>
      .bg { fill:#0b1020; }
      .grid { stroke:#24304f; stroke-width:1; }
      .board { fill:none; stroke:#3a476b; stroke-width:2; }
      .label { fill:#e5e7eb; font-family:system-ui,Segoe UI,Arial,sans-serif; font-size:13px; }
      .move { fill:none; stroke:#c7d2fe; stroke-width:2.5; marker-end:url(#arrow); filter:url(#glow); }
      .piece { font-size:36px; dominant-baseline:middle; text-anchor:middle; }
      .node { fill:#f8fafc; stroke:#111827; stroke-width:2; }
      .center { fill:#f59e0b; stroke:#111827; stroke-width:3; }
      .movehint { fill:#a78bfa; opacity:0.2; }
    </style>
  </defs>

  <rect class="bg" x="0" y="0" width="520" height="520"/>

  <!-- board -->
  <g transform="translate(60,60)">
    <rect class="board" x="0" y="0" width="400" height="400" rx="10"/>
    <!-- grid -->
    <g opacity="0.7">
      <path class="grid" d="M80 0V400M160 0V400M240 0V400M320 0V400"/>
      <path class="grid" d="M0 80H400M0 160H400M0 240H400M0 320H400"/>
    </g>

    <!-- zebra move from center -->
    <g>
      <!-- center square -->
      <circle class="center" cx="200" cy="200" r="16"/>
      <!-- leaper destinations (2,3) and (3,2) in 8 directions -->
      <g>
        <path class="move" d="M200 200 L120 80"/>
        <path class="move" d="M200 200 L280 80"/>
        <path class="move" d="M200 200 L80 120"/>
        <path class="move" d="M200 200 L320 120"/>
        <path class="move" d="M200 200 L80 280"/>
        <path class="move" d="M200 200 L320 280"/>
        <path class="move" d="M200 200 L120 320"/>
        <path class="move" d="M200 200 L280 320"/>
      </g>

      <!-- destination nodes -->
      <g>
        <circle class="node" cx="120" cy="80" r="11"/>
        <circle class="node" cx="280" cy="80" r="11"/>
        <circle class="node" cx="80" cy="120" r="11"/>
        <circle class="node" cx="320" cy="120" r="11"/>
        <circle class="node" cx="80" cy="280" r="11"/>
        <circle class="node" cx="320" cy="280" r="11"/>
        <circle class="node" cx="120" cy="320" r="11"/>
        <circle class="node" cx="280" cy="320" r="11"/>
      </g>

      <!-- zebra emojis -->
      <g class="piece">
        <text x="200" y="201">🦓</text>
        <text x="120" y="80">🦓</text>
        <text x="280" y="80">🦓</text>
        <text x="80" y="120">🦓</text>
        <text x="320" y="120">🦓</text>
        <text x="80" y="280">🦓</text>
        <text x="320" y="280">🦓</text>
        <text x="120" y="320">🦓</text>
        <text x="280" y="320">🦓</text>
      </g>
    </g>

    <!-- subtle move labels -->
    <g class="label" text-anchor="middle">
      <text x="200" y="28">Zebra graph moves: (2,3) leaper</text>
      <text x="200" y="388">8 possible jumps from any square</text>
    </g>
  </g>
</svg>
ZEBRA_END

In [ ]:
#% html
#my $conf = llm-configuration('ChatGPT', model => 'gpt-5.3-chat-latest');
my $conf = llm-configuration('ChatGPT', model => 'gpt-5.4-mini');
#my $conf = llm-configuration('Gemini', model => 'gemini-3.5-flash');
# llm-synthesize([
#         "Make instructive graphic of the moves of the so called Zebra graph.",
#         "That is a leaper graph with parameters (2, 3).",
#         "Use the zebra emoji.",
#         "Minimal image margins, dark mode.",
#         llm-prompt('NothingElse')('SVG')
#     ], 
#     e => $conf
# )

---

## Unit graph using complex numbers

```wolfram
p=Tuples[Range[-2,2],4] . I^{0,1,4/3,7/3};
RelationGraph[Abs[#1-#2]==1&,p,VertexCoordinates->ReIm@p]
```

In [ ]:
my @p = cross((-2 ... 2) xx 4).map({ dot-product($_, i <<**>> (0, 1, 4/3, 7/3)) });
my $g = Graph::Relation.new({abs(abs(@p[$^a] - @p[$^b]) - 1) ≤ 1e-8}, ^@p.elems, as => {.Str}, vertex-coordinates => @p.kv.map(-> $k, $v { $k => [$v.re, $v.im]}).Hash);
$g

In [ ]:
#% js
js-d3-graph-plot(
    $g.edges,
    vertex-coordinates => $g.vertex-coordinates,
    background => 'none',
    vertex-label-color => 'none',
    vertex-fill-color => 'purple',
    vertex-size => 4,
    edge-thickness => 0.7,
    :800width,
    :800height,
)

In [ ]:
#% js
js-d3-list-plot(
    $g.vertex-coordinates.values,
    :$background,
    :400width,
    :400height,
)

In [ ]:
my %smaller = $g.vertex-coordinates.grep({ norm($_.value) ≤ 1.5});
deduce-type(%smaller)

In [ ]:
#% js
$g.subgraph(%smaller.keys).edges
==> js-d3-graph-plot(
    vertex-coordinates => %smaller,
    background => 'none',
    vertex-label-color => 'none',
    vertex-fill-color => 'purple',
    vertex-size => 4,
    edge-thickness => 0.7,
    :800width,
    :800height,
)

In [ ]:
my @triplets = $g.adjacency-list.kv.map( -> $k, %v { %v.map({ [$k.Int, $_.key.Int, $_.value] }) }).flat(1);
deduce-type(@triplets)

In [ ]:
my @mat = $g.subgraph(%smaller.keys).distance-matrix;
deduce-type(@mat)

In [ ]:
my @mat = $g.adjacency-matrix;
deduce-type(@mat)

In [ ]:
#% js
js-d3-matrix-plot(@triplets.map({ <x y z>.Array Z=> $_.Array })».Hash, :!mesh, :!grid-lines, :!tooltip, color-palette => 'Magma')

In [ ]:
#% html
$g.dot(
    engine => 'neato', 
    :8graph-size, 
    :!node-labels, 
    vertex-shape => 'point', 
    vertex-width => 0.1, 
    vertex-color => 'purple',
    vertex-fill-color => 'purple',
    edge-thickness => 0.5
):svg